In [1]:
import pandas as pd

In [2]:
train = pd.read_csv("../data/processed/train.csv")

In [3]:
cols = list(train.columns)
URL_COLS = cols[cols.index("length_url") : cols.index("statistical_report") + 1]
CONTENT_COLS = cols[cols.index("nb_hyperlinks") : cols.index("domain_with_copyright") + 1]
EXTERNAL_COLS = cols[cols.index("whois_registered_domain") : cols.index("page_rank") + 1]

print(len(URL_COLS), len(CONTENT_COLS), len(EXTERNAL_COLS))

56 24 7


In [4]:
X = train[URL_COLS + CONTENT_COLS + EXTERNAL_COLS]
y = (train["status"] == "phishing").astype(int)   # 1 = phishing
groups = train["group"]

print(X.shape, y.mean().round(3), groups.nunique())

(9003, 87) 0.492 5948


In [5]:
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier


In [6]:
cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)

In [7]:
scores = cross_validate(rf, X[URL_COLS], y, groups=groups, cv=cv,scoring=["f1", "precision", "recall", "roc_auc"])

In [8]:
pd.DataFrame(scores).drop(columns=["fit_time", "score_time"]).agg(["mean", "std"]).round(4)

,test_f1,test_precision,test_recall,test_roc_auc
mean,0.8784,0.8729,0.8843,0.9521
std,0.0246,0.0283,0.0250,0.0125


In [9]:
from sklearn.model_selection import KFold

In [10]:
cv_random = KFold(n_splits=5, shuffle=True, random_state=42)

In [11]:
scores_random = cross_validate(rf, X[URL_COLS], y, cv=cv_random,scoring=["f1", "precision", "recall", "roc_auc"])

In [12]:
comparison = pd.DataFrame({
    "par site (GroupKFold)": pd.DataFrame(scores).mean(),
    "aléatoire (KFold)": pd.DataFrame(scores_random).mean(),
}).drop(["fit_time", "score_time"]).round(4)
comparison["écart"] = comparison["aléatoire (KFold)"] - comparison["par site (GroupKFold)"]
comparison

,par site (GroupKFold),aléatoire (KFold),écart
test_f1,0.8784,0.9082,0.0298
test_precision,0.8729,0.9112,0.0383
test_recall,0.8843,0.9053,0.0210
test_roc_auc,0.9521,0.9675,0.0154
